In [1]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import MACCSkeys, Descriptors,AllChem
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem.Descriptors import MolWt
from rdkit.Chem.rdMolDescriptors import CalcTPSA
import numpy as np

In [4]:
solubility_df = pd.read_excel('C:/Users/kverg/GDI-NN/data/solubility_cosolvents/Bao_Allen_raw.xlsx', sheet_name='Solubility')
drugs_df = pd.read_excel('C:/Users/kverg/GDI-NN/data/solubility_cosolvents/Bao_Allen_raw.xlsx', sheet_name='Drugs')
solvents_df = pd.read_excel('C:/Users/kverg/GDI-NN/data/solubility_cosolvents/Bao_Allen_raw.xlsx', sheet_name='Solvents')
solubility_df

,Web of Science Index,Drug,Solvent_1,Solvent_1_weight_fraction,Solvent_1_mol_fraction,Solvent_2,Temperature (K),Solubility (mol/mol),DOI
0,36,Guanidine hydrochloride,Dimethylformamide,0.1001,NaN,1-Propanol,278.15,0.069400,doi.org/10.1016/j.molliq.2023.122902
1,36,Guanidine hydrochloride,Dimethylformamide,0.1001,NaN,1-Propanol,283.15,0.077050,doi.org/10.1016/j.molliq.2023.122902
2,36,Guanidine hydrochloride,Dimethylformamide,0.1001,NaN,1-Propanol,288.15,0.084960,doi.org/10.1016/j.molliq.2023.122902
3,36,Guanidine hydrochloride,Dimethylformamide,0.1001,NaN,1-Propanol,293.15,0.090850,doi.org/10.1016/j.molliq.2023.122902
4,36,Guanidine hydrochloride,Dimethylformamide,0.1001,NaN,1-Propanol,298.15,0.097740,doi.org/10.1016/j.molliq.2023.122902
...,...,...,...,...,...,...,...,...,...
28093,Lab,Aspirin,Ethanol,0.5000,NaN,Water,298.15,0.015803,Lab
28094,Lab,Aspirin,Ethanol,0.8000,NaN,Water,298.15,0.046089,Lab
28095,Lab,Aspirin,Ethanol,0.2000,NaN,Water,313.15,0.002981,Lab
28096,Lab,Aspirin,Ethanol,0.5000,NaN,Water,313.15,0.029540,Lab


In [5]:
lab = solubility_df[solubility_df['DOI'] == 'Lab']
literature = solubility_df[solubility_df['DOI'] != 'Lab']
literature.shape

(28074, 9)

In [6]:
def duplicate_removal(df):
    
    df['Mono solvent'] = 'No'

    df.loc[(df['Solvent_1_weight_fraction'] == 1) | (df['Solvent_1_mol_fraction'] == 1), 'Mono solvent'] = df['Solvent_1']
    df.loc[(df['Solvent_1_weight_fraction'] == 0) | (df['Solvent_1_mol_fraction'] == 0), 'Mono solvent'] = df['Solvent_2']

    df_no_duplicates = df[df['Mono solvent'] != 'No'].drop_duplicates(subset=['Drug', 'Mono solvent', 'Temperature (K)'])

    result_df = pd.concat([df[df['Mono solvent'] == 'No'], df_no_duplicates], ignore_index=True)
    
    result_df = result_df.drop(['Mono solvent'], axis = 1)

    return result_df

literature = duplicate_removal(literature)
literature.shape

C:\Users\kverg\AppData\Local\Temp\ipykernel_33156\525658986.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Mono solvent'] = 'No'


(26706, 9)

In [7]:
literature['Solvent_1_Fraction'] = literature['Solvent_1_weight_fraction'].combine_first(literature['Solvent_1_mol_fraction'])
literature['Solvent_2_Fraction'] = 1 - literature['Solvent_1_Fraction']

In [8]:
def check_and_swap(group):
    correlation_1 = group['Solvent_1_Fraction'].corr(group['Solubility (mol/mol)'])
    correlation_2 = group['Solvent_2_Fraction'].corr(group['Solubility (mol/mol)'])

    if correlation_1 < correlation_2:

        temp_solvent = group['Solvent_1'].copy()
        group['Solvent_1'] = group['Solvent_2']
        group['Solvent_2'] = temp_solvent


        group['Solvent_1_Fraction'] = 1 - group['Solvent_1_Fraction']
        

        if 'Solvent_1_weight_fraction' in group and group['Solvent_1_weight_fraction'].notna().all():
            group['Solvent_1_weight_fraction'] = 1 - group['Solvent_1_weight_fraction']

        if 'Solvent_1_mol_fraction' in group and group['Solvent_1_mol_fraction'].notna().all():
            group['Solvent_1_mol_fraction'] = 1 - group['Solvent_1_mol_fraction']

    return group


literature = literature.groupby(['Drug', 'Solvent_1', 'Solvent_2', 'Temperature (K)']).apply(check_and_swap).reset_index(drop=True)

c:\Users\kverg\miniforge3\envs\GDINN1\lib\site-packages\numpy\lib\_function_base_impl.py:2914: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
c:\Users\kverg\miniforge3\envs\GDINN1\lib\site-packages\numpy\lib\_function_base_impl.py:2773: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
c:\Users\kverg\miniforge3\envs\GDINN1\lib\site-packages\numpy\lib\_function_base_impl.py:2773: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)
C:\Users\kverg\AppData\Local\Temp\ipykernel_33156\3285027337.py:24: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  literature = literature.groupby(['Drug', 'Solvent_

In [9]:
literature = literature.drop(['Solvent_1_Fraction', 'Solvent_2_Fraction'], axis = 1)
duplicates_mask = literature.duplicated(subset=['Drug', 'Solvent_1', 'Solvent_1_weight_fraction', 'Solvent_1_mol_fraction', 'Solvent_2', 'Temperature (K)'], keep=False)

df_duplicates = literature[duplicates_mask]

df_duplicates = df_duplicates.reset_index(drop=True)

df_duplicates_sorted = df_duplicates.sort_values(by=['Drug', 'Solvent_1', 'Solvent_1_weight_fraction', 'Solvent_1_mol_fraction', 'Solvent_2', 'Temperature (K)']).reset_index(drop=True)

df_duplicates_sorted.shape[0]

0

In [10]:
def find_diastereomers(df,Compound):
    possible_diastereomers = []
    
    for index, row in df.iterrows():
        smiles = row['SMILES']
        #smiles = row['SMILES']
        drug_name = row[Compound]
        

        mol = Chem.MolFromSmiles(smiles)
        
        if mol:
            chiral_centers = Chem.FindMolChiralCenters(mol, includeUnassigned=True)
            

            if len(chiral_centers) > 1:
                possible_diastereomers.append(drug_name)
    
    return possible_diastereomers

In [11]:
possible_diastereomers = find_diastereomers(drugs_df, 'Drug')
print("possible_diastereomers:")
for drug in possible_diastereomers:
    print(drug)

possible_diastereomers:
Glucosamine hydrochloride
Itraconazole
5-Azacytidine
Florfenicol
Sofosbuvir
Posaconazole
Gestodene
Labetalol hydrochloride
Oleanolic acid
Ursolic acid
Trans-4-Hydroxy‐L‐proline
Pidotimod
D(−)-Salicin
Etonogestrel
Ketoconazole
D‐Ribose
Lanosterol
Capecitabine
Griseofulvin
Azlocillin
Rivastigmine tartrate
Artesunate
Vinpocetine
Hydrocortisone


In [12]:
possible_diastereomers = find_diastereomers(solvents_df, 'Solvent')
print("possible_diastereomers:")
for solvent in possible_diastereomers:
    print(solvent)

possible_diastereomers:


In [13]:
tautomer_enumerator = rdMolStandardize.TautomerEnumerator()
uncharger = rdMolStandardize.Uncharger()

def standardize_smiles(smiles):
    mol = Chem.MolFromSmiles(smiles)
    mol = tautomer_enumerator.Canonicalize(mol)
    mol = uncharger.uncharge(mol)
    return Chem.MolToSmiles(mol)

In [14]:
drugs_df['standardized_SMILES'] = drugs_df['SMILES'].apply(standardize_smiles)

[12:22:19] Running Uncharger
[12:22:19] Running Uncharger
[12:22:19] Running Uncharger
[12:22:19] Running Uncharger
[12:22:19] Running Uncharger
[12:22:19] Running Uncharger
[12:22:19] Running Uncharger
[12:22:19] Running Uncharger
[12:22:19] Running Uncharger
[12:22:19] Running Uncharger
[12:22:19] Running Uncharger
[12:22:19] Running Uncharger
[12:22:19] Running Uncharger
[12:22:19] Running Uncharger
[12:22:19] Running Uncharger
[12:22:19] Running Uncharger
[12:22:19] Running Uncharger
[12:22:19] Running Uncharger
[12:22:19] Running Uncharger
[12:22:19] Running Uncharger
[12:22:19] Running Uncharger
[12:22:19] Running Uncharger
[12:22:19] Running Uncharger
[12:22:19] Running Uncharger
[12:22:19] Running Uncharger
[12:22:19] Running Uncharger
[12:22:19] Running Uncharger
[12:22:19] Running Uncharger
[12:22:19] Running Uncharger
[12:22:19] Running Uncharger
[12:22:19] Running Uncharger
[12:22:19] Running Uncharger
[12:22:19] Running Uncharger
[12:22:19] Can't kekulize mol.  Unkekulized

In [15]:
solvents_df['standardized_SMILES'] = solvents_df['SMILES'].apply(standardize_smiles)

[12:22:19] Running Uncharger
[12:22:19] Running Uncharger
[12:22:19] Running Uncharger
[12:22:19] Running Uncharger
[12:22:19] Running Uncharger
[12:22:19] Running Uncharger
[12:22:19] Running Uncharger
[12:22:19] Running Uncharger
[12:22:19] Running Uncharger
[12:22:19] Running Uncharger
[12:22:19] Running Uncharger
[12:22:19] Running Uncharger
[12:22:19] Running Uncharger
[12:22:19] Running Uncharger
[12:22:19] Running Uncharger
[12:22:19] Running Uncharger
[12:22:19] Running Uncharger
[12:22:19] Running Uncharger
[12:22:19] Running Uncharger
[12:22:19] Running Uncharger
[12:22:19] Running Uncharger
[12:22:19] Running Uncharger
[12:22:19] Running Uncharger
[12:22:19] Running Uncharger
[12:22:19] Running Uncharger
[12:22:19] Running Uncharger
[12:22:19] Running Uncharger
[12:22:19] Running Uncharger
[12:22:19] Running Uncharger
[12:22:19] Running Uncharger
[12:22:19] Running Uncharger
[12:22:19] Running Uncharger
[12:22:19] Running Uncharger
[12:22:19] Running Uncharger
[12:22:19] Run

In [16]:
def enhance_solubility_data(solubility_df, drugs_df, solvents_df, drug_features, solvent_features):

    drug_col_rename_map = {feature: f"Drug_{feature}" for feature in drug_features}
    selected_drug_features = drugs_df[drug_features + ['Drug']]
    renamed_drug_features = selected_drug_features.rename(columns=drug_col_rename_map)
    enhanced_df = solubility_df.merge(renamed_drug_features, on='Drug', how='left')

    solvent_1_col_rename_map = {feature: f"Solvent_1_{feature}" for feature in solvent_features}
    solvent_1_col_rename_map['Solvent'] = 'Solvent_1'
    selected_solvent_1_features = solvents_df[solvent_features + ['Solvent']].rename(columns=solvent_1_col_rename_map)
    enhanced_df = enhanced_df.merge(selected_solvent_1_features, left_on='Solvent_1', right_on='Solvent_1', how='left')

    solvent_2_col_rename_map = {feature: f"Solvent_2_{feature}" for feature in solvent_features}
    solvent_2_col_rename_map['Solvent'] = 'Solvent_2'
    selected_solvent_2_features = solvents_df[solvent_features + ['Solvent']].rename(columns=solvent_2_col_rename_map)
    enhanced_df = enhanced_df.merge(selected_solvent_2_features, left_on='Solvent_2', right_on='Solvent_2', how='left')

    return enhanced_df

In [27]:
from rdkit.Chem import MolFromSmiles as smi2mol
from rdkit.Chem import MolToSmiles as mol2smi
## Function to create canonical smiles 
def canon(smi):
    try:
        mol=smi2mol(smi, sanitize=True)
        smi_canon=mol2smi(mol, isomericSmiles=False, canonical=True)
        return(smi_canon)
    except:
        print("ERROR")
        return(smi)
    
#### Applying function to create the column with canonical smiles.  
solvents_df['Canonical_smiles'] = [canon(smi) for smi in solvents_df.SMILES]


In [37]:
def generate_feats(df):
    example = Chem.MolFromSmiles('C')
    example = Chem.AddHs(example)
    AllChem.EmbedMolecule(example)
    
    descriptors_list = []
    
    descriptor_names = [desc[0] for desc in Descriptors._descList]
    descriptor_functions = [desc[1] for desc in Descriptors._descList]

    for smile in df['Canonical_smiles']:

        mol = Chem.MolFromSmiles(smile)
        
        mol_3d = Chem.AddHs(mol)
        AllChem.EmbedMolecule(mol_3d)
        
        descriptor_values = [func(mol) for func in descriptor_functions]
        descriptors_list.append(descriptor_values)

    descriptors_df = pd.DataFrame(descriptors_list, columns=[name for name in descriptor_names])

    combined_df = pd.concat([df, descriptors_df], axis=1)

    return combined_df

In [44]:
drug_feats = generate_feats(drugs_df)
drug_feats

,Drug,Drugs@FDA,CAS,SMILES,Canonical_smiles,CAS_MP (C),Chemical_book_MP (C),ChemSpider_MP (C),SCBT_MP (C),Sigma_MP (C),...,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea
0,Guanidine hydrochloride,Yes,50-01-1,C(=N)(N)N.Cl,Cl.N=C(N)N,178.0,182.5,NaN,187.0,NaN,...,0,0,0,0,0,0,0,0,0,0
1,Glucosamine hydrochloride,No,66-84-2,[C@H]([C@@H]([C@@H](CO)O)O)([C@H](C=O)N)O.Cl,Cl.O=CC(N)C(O)C(O)C(O)CO,NaN,NaN,NaN,192.0,192.0,...,0,0,0,0,0,0,0,0,0,0
2,2-Amino-6-chloropyrazine,No,33332-28-4,ClC1=NC(N)=CN=C1,ClC=1N=C(N)C=NC1,153.0,NaN,151.00,145.0,NaN,...,0,0,0,0,0,0,0,0,0,0
3,Thiamine nitrate,No,532-43-4,N(=O)(=O)[O-].C([N+]=1C(C)=C(CCO)SC1)C=2C(N)=N...,O=N(=O)[O-].OCCC=1SC=[N+](C1C)CC2=CN=C(N=C2N)C,198.0,NaN,NaN,190.0,NaN,...,0,0,0,0,0,1,0,0,0,0
4,Aripiprazole,Yes,129722-12-9,O(CCCCN1CCN(CC1)C2=C(Cl)C(Cl)=CC=C2)C=3C=C4C(=...,O=C1NC2=CC(OCCCCN3CCN(C=4C=CC=C(Cl)C4Cl)CC3)=C...,NaN,NaN,139.25,140.0,NaN,...,0,0,0,0,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
121,Carbendazim,No,10605-21-7,N(C(OC)=O)C=1NC=2C(N1)=CC=CC2,O=C(OC)NC1=NC=2C=CC=CC2N1,300.0,NaN,NaN,300.0,NaN,...,0,0,0,0,0,0,0,0,0,0
122,Vinpocetine,No,42971-09-5,C(C)[C@]12[C@]3(C=4N(C=5C(C4CCN3CCC1)=CC=CC5)C...,O=C(OCC)C1=CC2(CC)CCCN3CCC=4C=5C=CC=CC5N1C4C32,150.0,NaN,NaN,148.0,NaN,...,0,0,0,0,0,0,0,0,0,0
123,3-Indolepropionic acid,No,830-96-6,C(CC(O)=O)C=1C=2C(NC1)=CC=CC2,O=C(O)CCC1=CNC=2C=CC=CC21,134.5,NaN,NaN,134.5,NaN,...,0,0,0,0,0,0,0,0,0,0
124,Hydrocortisone,Yes,50-23-7,C[C@@]12[C@]([C@]3([C@@]([C@]4(C)C(CC3)=CC(=O)...,O=C1C=C2CCC3C4CCC(O)(C(=O)CO)C4(C)CC(O)C3C2(C)CC1,218.5,NaN,NaN,212.5,NaN,...,0,0,0,0,0,0,0,0,0,0


In [45]:
solvent_feats = generate_feats(solvents_df)
solvent_feats

,Solvent,CAS,isomeric_SMILES,SMILES,CAS_MP (C),SCBT_MP (C),Sigma_MP (C),Fisher_MP (C),Wiki_MP (C),Collected_Melting_temp (C),...,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea
0,Dimethylformamide,68-12-2,N(C=O)(C)C,O=CN(C)C,-61.00,-61.0,-61.0,NaN,NaN,-61.000000,...,0,0,0,0,0,0,0,0,0,0
1,2-Methoxyethanol,109-86-4,C(CO)OC,OCCOC,-85.10,-85.0,-85.0,NaN,NaN,-85.033333,...,0,0,0,0,0,0,0,0,0,0
2,Methanol,67-56-1,CO,OC,-97.80,NaN,-98.0,-98.0,NaN,-97.933333,...,0,0,0,0,0,0,0,0,0,0
3,N-methyl-2-pyrrolidone,872-50-4,O=C1N(C)CCC1,O=C1N(C)CCC1,-25.00,-24.0,-24.0,NaN,NaN,-24.333333,...,0,0,0,0,0,0,0,0,0,0
4,Water,7732-18-5,O,O,0.00,NaN,0.0,0.0,NaN,0.000000,...,0,0,0,0,0,0,0,0,0,0
5,Ethanol,64-17-5,C(C)O,OCC,-114.10,NaN,-114.0,NaN,-114.14,-114.080000,...,0,0,0,0,0,0,0,0,0,0
6,1-Propanol,71-23-8,C(CO)C,OCCC,-127.00,-127.0,-127.0,NaN,NaN,-127.000000,...,0,0,0,0,0,0,0,0,0,0
7,Propylene glycol,57-55-6,C(CO)(C)O,OCC(O)C,-59.00,-59.0,-60.0,NaN,NaN,-59.333333,...,0,0,0,0,0,0,0,0,0,0
8,Acetone,67-64-1,C(C)(C)=O,O=C(C)C,-94.80,NaN,-94.0,-95.0,NaN,-94.600000,...,0,0,0,0,0,0,0,0,0,0
9,2-Propanol,67-63-0,C(C)(C)O,OC(C)C,-88.50,-89.5,-89.5,NaN,NaN,-89.166667,...,0,0,0,0,0,0,0,0,0,0


In [46]:
rdkit_feats = [desc[0] for desc in Descriptors._descList]
drug_feats_to_extract = ['Canonical_smiles'] + rdkit_feats
solvent_feats_to_extract = ['Canonical_smiles'] + rdkit_feats
enhanced_literature = enhance_solubility_data(literature, drug_feats, solvent_feats, drug_feats_to_extract, solvent_feats_to_extract)
enhanced_lab = enhance_solubility_data(lab, drug_feats, solvent_feats, drug_feats_to_extract, solvent_feats_to_extract)
enhanced_lab

,Web of Science Index,Drug,Solvent_1,Solvent_1_weight_fraction,Solvent_1_mol_fraction,Solvent_2,Temperature (K),Solubility (mol/mol),DOI,Drug_Canonical_smiles,...,Solvent_2_fr_sulfide,Solvent_2_fr_sulfonamd,Solvent_2_fr_sulfone,Solvent_2_fr_term_acetylene,Solvent_2_fr_tetrazole,Solvent_2_fr_thiazole,Solvent_2_fr_thiocyan,Solvent_2_fr_thiophene,Solvent_2_fr_unbrch_alkane,Solvent_2_fr_urea
0,Lab,Celecoxib,Ethanol,0.2,NaN,Water,298.15,4.674341e-07,Lab,O=S(=O)(N)C1=CC=C(C=C1)N2N=C(C=C2C=3C=CC(=CC3)...,...,0,0,0,0,0,0,0,0,0,0
1,Lab,Celecoxib,Ethanol,0.5,NaN,Water,298.15,3.039692e-04,Lab,O=S(=O)(N)C1=CC=C(C=C1)N2N=C(C=C2C=3C=CC(=CC3)...,...,0,0,0,0,0,0,0,0,0,0
2,Lab,Celecoxib,Ethanol,0.8,NaN,Water,298.15,4.666784e-03,Lab,O=S(=O)(N)C1=CC=C(C=C1)N2N=C(C=C2C=3C=CC(=CC3)...,...,0,0,0,0,0,0,0,0,0,0
3,Lab,Celecoxib,Ethanol,0.2,NaN,Water,313.15,1.679215e-06,Lab,O=S(=O)(N)C1=CC=C(C=C1)N2N=C(C=C2C=3C=CC(=CC3)...,...,0,0,0,0,0,0,0,0,0,0
4,Lab,Celecoxib,Ethanol,0.5,NaN,Water,313.15,7.044887e-04,Lab,O=S(=O)(N)C1=CC=C(C=C1)N2N=C(C=C2C=3C=CC(=CC3)...,...,0,0,0,0,0,0,0,0,0,0
5,Lab,Celecoxib,Ethanol,0.8,NaN,Water,313.15,8.569214e-03,Lab,O=S(=O)(N)C1=CC=C(C=C1)N2N=C(C=C2C=3C=CC(=CC3)...,...,0,0,0,0,0,0,0,0,0,0
6,Lab,Acetaminophen,Ethanol,0.2,NaN,Water,298.15,5.187946e-03,Lab,O=C(NC1=CC=C(O)C=C1)C,...,0,0,0,0,0,0,0,0,0,0
7,Lab,Acetaminophen,Ethanol,0.5,NaN,Water,298.15,2.854515e-02,Lab,O=C(NC1=CC=C(O)C=C1)C,...,0,0,0,0,0,0,0,0,0,0
8,Lab,Acetaminophen,Ethanol,0.8,NaN,Water,298.15,5.196241e-02,Lab,O=C(NC1=CC=C(O)C=C1)C,...,0,0,0,0,0,0,0,0,0,0
9,Lab,Acetaminophen,Ethanol,0.2,NaN,Water,313.15,9.748131e-03,Lab,O=C(NC1=CC=C(O)C=C1)C,...,0,0,0,0,0,0,0,0,0,0


In [47]:
def calculate_fractions(df):
    for index, row in df.iterrows():

        molecular_weight_1 = row['Solvent_1_ExactMolWt']
        molecular_weight_2 = row['Solvent_2_ExactMolWt']

        if pd.isna(row['Solvent_1_mol_fraction']) and not pd.isna(row['Solvent_1_weight_fraction']):
            weight_fraction_1 = float(row['Solvent_1_weight_fraction'])
            weight_fraction_2 = 1 - weight_fraction_1
            mole_fraction_1 = (weight_fraction_1 / molecular_weight_1) / ((weight_fraction_1 / molecular_weight_1) + (weight_fraction_2 / molecular_weight_2))
            df.at[index, 'Solvent_1_mol_fraction'] = mole_fraction_1

        elif not pd.isna(row['Solvent_1_mol_fraction']) and pd.isna(row['Solvent_1_weight_fraction']):
            mole_fraction_1 = float(row['Solvent_1_mol_fraction'])
            mole_fraction_2 = 1 - mole_fraction_1
            weight_fraction_1 = (mole_fraction_1 * molecular_weight_1) / ((mole_fraction_1 * molecular_weight_1) + (mole_fraction_2 * molecular_weight_2))
            df.at[index, 'Solvent_1_weight_fraction'] = weight_fraction_1
            
    return df


enhanced_literature = calculate_fractions(enhanced_literature)
enhanced_lab = calculate_fractions(enhanced_lab)
enhanced_lab

,Web of Science Index,Drug,Solvent_1,Solvent_1_weight_fraction,Solvent_1_mol_fraction,Solvent_2,Temperature (K),Solubility (mol/mol),DOI,Drug_Canonical_smiles,...,Solvent_2_fr_sulfide,Solvent_2_fr_sulfonamd,Solvent_2_fr_sulfone,Solvent_2_fr_term_acetylene,Solvent_2_fr_tetrazole,Solvent_2_fr_thiazole,Solvent_2_fr_thiocyan,Solvent_2_fr_thiophene,Solvent_2_fr_unbrch_alkane,Solvent_2_fr_urea
0,Lab,Celecoxib,Ethanol,0.2,0.089083,Water,298.15,4.674341e-07,Lab,O=S(=O)(N)C1=CC=C(C=C1)N2N=C(C=C2C=3C=CC(=CC3)...,...,0,0,0,0,0,0,0,0,0,0
1,Lab,Celecoxib,Ethanol,0.5,0.281185,Water,298.15,3.039692e-04,Lab,O=S(=O)(N)C1=CC=C(C=C1)N2N=C(C=C2C=3C=CC(=CC3)...,...,0,0,0,0,0,0,0,0,0,0
2,Lab,Celecoxib,Ethanol,0.8,0.610093,Water,298.15,4.666784e-03,Lab,O=S(=O)(N)C1=CC=C(C=C1)N2N=C(C=C2C=3C=CC(=CC3)...,...,0,0,0,0,0,0,0,0,0,0
3,Lab,Celecoxib,Ethanol,0.2,0.089083,Water,313.15,1.679215e-06,Lab,O=S(=O)(N)C1=CC=C(C=C1)N2N=C(C=C2C=3C=CC(=CC3)...,...,0,0,0,0,0,0,0,0,0,0
4,Lab,Celecoxib,Ethanol,0.5,0.281185,Water,313.15,7.044887e-04,Lab,O=S(=O)(N)C1=CC=C(C=C1)N2N=C(C=C2C=3C=CC(=CC3)...,...,0,0,0,0,0,0,0,0,0,0
5,Lab,Celecoxib,Ethanol,0.8,0.610093,Water,313.15,8.569214e-03,Lab,O=S(=O)(N)C1=CC=C(C=C1)N2N=C(C=C2C=3C=CC(=CC3)...,...,0,0,0,0,0,0,0,0,0,0
6,Lab,Acetaminophen,Ethanol,0.2,0.089083,Water,298.15,5.187946e-03,Lab,O=C(NC1=CC=C(O)C=C1)C,...,0,0,0,0,0,0,0,0,0,0
7,Lab,Acetaminophen,Ethanol,0.5,0.281185,Water,298.15,2.854515e-02,Lab,O=C(NC1=CC=C(O)C=C1)C,...,0,0,0,0,0,0,0,0,0,0
8,Lab,Acetaminophen,Ethanol,0.8,0.610093,Water,298.15,5.196241e-02,Lab,O=C(NC1=CC=C(O)C=C1)C,...,0,0,0,0,0,0,0,0,0,0
9,Lab,Acetaminophen,Ethanol,0.2,0.089083,Water,313.15,9.748131e-03,Lab,O=C(NC1=CC=C(O)C=C1)C,...,0,0,0,0,0,0,0,0,0,0


In [50]:
def calculate_logs(df_updated):
    
    total_moles = 1

    df_updated['mol0'] = total_moles * df_updated['Solubility (mol/mol)']
    df_updated['mol1'] = (total_moles - df_updated['mol0']) * df_updated['Solvent_1_mol_fraction']
    df_updated['mol2'] = (total_moles - df_updated['mol0']) * (1 - df_updated['Solvent_1_mol_fraction'])
    df_updated['total_mol'] = df_updated['mol0'] + df_updated['mol1'] + df_updated['mol2']
    print(df_updated[['total_mol']].describe())

    df_updated['mass0'] = df_updated['mol0'] * df_updated['Drug_ExactMolWt']
    df_updated['mass1'] = df_updated['mol1'] * df_updated['Solvent_1_ExactMolWt']
    df_updated['mass2'] = df_updated['mol2'] * df_updated['Solvent_2_ExactMolWt']
    df_updated['total_mass'] = df_updated['mass0'] + df_updated['mass1'] + df_updated['mass2']

    df_updated['Solubility (g/g)'] = df_updated['mass0'] / df_updated['total_mass']
    df_updated['Solubility (g/100g)'] = df_updated['Solubility (g/g)'] * 100
    df_updated['LogS'] = np.log10(df_updated['Solubility (g/100g)'])

    df_updated = df_updated.drop(['mol0', 'mol1', 'mol2', 'mass0','mass1','mass2','total_mass','total_mol','Solubility (g/g)'], axis = 1)
    
    return df_updated

updated_literature = calculate_logs(enhanced_literature)
updated_lab = calculate_logs(enhanced_lab)
updated_lab

          total_mol
count  2.670600e+04
mean   1.000000e+00
std    3.807570e-17
min    1.000000e+00
25%    1.000000e+00
50%    1.000000e+00
75%    1.000000e+00
max    1.000000e+00
          total_mol
count  2.400000e+01
mean   1.000000e+00
std    4.009654e-17
min    1.000000e+00
25%    1.000000e+00
50%    1.000000e+00
75%    1.000000e+00
max    1.000000e+00


,Web of Science Index,Drug,Solvent_1,Solvent_1_weight_fraction,Solvent_1_mol_fraction,Solvent_2,Temperature (K),Solubility (mol/mol),DOI,Drug_Canonical_smiles,...,Solvent_2_fr_sulfone,Solvent_2_fr_term_acetylene,Solvent_2_fr_tetrazole,Solvent_2_fr_thiazole,Solvent_2_fr_thiocyan,Solvent_2_fr_thiophene,Solvent_2_fr_unbrch_alkane,Solvent_2_fr_urea,Solubility (g/100g),LogS
0,Lab,Celecoxib,Ethanol,0.2,0.089083,Water,298.15,4.674341e-07,Lab,O=S(=O)(N)C1=CC=C(C=C1)N2N=C(C=C2C=3C=CC(=CC3)...,...,0,0,0,0,0,0,0,0,0.000869,-3.061188
1,Lab,Celecoxib,Ethanol,0.5,0.281185,Water,298.15,3.039692e-04,Lab,O=S(=O)(N)C1=CC=C(C=C1)N2N=C(C=C2C=3C=CC(=CC3)...,...,0,0,0,0,0,0,0,0,0.445512,-0.351141
2,Lab,Celecoxib,Ethanol,0.8,0.610093,Water,298.15,4.666784e-03,Lab,O=S(=O)(N)C1=CC=C(C=C1)N2N=C(C=C2C=3C=CC(=CC3)...,...,0,0,0,0,0,0,0,0,4.842238,0.685046
3,Lab,Celecoxib,Ethanol,0.2,0.089083,Water,313.15,1.679215e-06,Lab,O=S(=O)(N)C1=CC=C(C=C1)N2N=C(C=C2C=3C=CC(=CC3)...,...,0,0,0,0,0,0,0,0,0.003120,-2.505811
4,Lab,Celecoxib,Ethanol,0.5,0.281185,Water,313.15,7.044887e-04,Lab,O=S(=O)(N)C1=CC=C(C=C1)N2N=C(C=C2C=3C=CC(=CC3)...,...,0,0,0,0,0,0,0,0,1.026914,0.011534
5,Lab,Celecoxib,Ethanol,0.8,0.610093,Water,313.15,8.569214e-03,Lab,O=S(=O)(N)C1=CC=C(C=C1)N2N=C(C=C2C=3C=CC(=CC3)...,...,0,0,0,0,0,0,0,0,8.576122,0.933291
6,Lab,Acetaminophen,Ethanol,0.2,0.089083,Water,298.15,5.187946e-03,Lab,O=C(NC1=CC=C(O)C=C1)C,...,0,0,0,0,0,0,0,0,3.699358,0.568126
7,Lab,Acetaminophen,Ethanol,0.5,0.281185,Water,298.15,2.854515e-02,Lab,O=C(NC1=CC=C(O)C=C1)C,...,0,0,0,0,0,0,0,0,14.634461,1.165377
8,Lab,Acetaminophen,Ethanol,0.8,0.610093,Water,298.15,5.196241e-02,Lab,O=C(NC1=CC=C(O)C=C1)C,...,0,0,0,0,0,0,0,0,19.081476,1.280612
9,Lab,Acetaminophen,Ethanol,0.2,0.089083,Water,313.15,9.748131e-03,Lab,O=C(NC1=CC=C(O)C=C1)C,...,0,0,0,0,0,0,0,0,6.761074,0.830016


In [52]:
print(updated_literature.shape)
print(updated_lab.shape)
print(updated_lab.columns)

(26706, 644)
(24, 644)
Index(['Web of Science Index', 'Drug', 'Solvent_1',
       'Solvent_1_weight_fraction', 'Solvent_1_mol_fraction', 'Solvent_2',
       'Temperature (K)', 'Solubility (mol/mol)', 'DOI',
       'Drug_Canonical_smiles',
       ...
       'Solvent_2_fr_sulfone', 'Solvent_2_fr_term_acetylene',
       'Solvent_2_fr_tetrazole', 'Solvent_2_fr_thiazole',
       'Solvent_2_fr_thiocyan', 'Solvent_2_fr_thiophene',
       'Solvent_2_fr_unbrch_alkane', 'Solvent_2_fr_urea',
       'Solubility (g/100g)', 'LogS'],
      dtype='object', length=644)


In [55]:
columns_to_keep = ['Drug','Solvent_1','Solvent_2','Solvent_1_mol_fraction','Drug_Canonical_smiles',
                    'Solvent_1_Canonical_smiles', 'Solvent_2_Canonical_smiles', 'Temperature (K)','LogS']
final_literature = updated_literature[columns_to_keep]
final_literature

,Drug,Solvent_1,Solvent_2,Solvent_1_mol_fraction,Drug_Canonical_smiles,Solvent_1_Canonical_smiles,Solvent_2_Canonical_smiles,Temperature (K),LogS
0,"1,1-Diamino-2,2-dinitroethylene",Dimethyl sulfoxide,4-Methyl-2-pentanone,0.9406,O=N(=O)C(=C(N)N)N(=O)=O,CS(C)=O,CC(=O)CC(C)C,293.15,1.437461
1,"1,1-Diamino-2,2-dinitroethylene",Dimethyl sulfoxide,4-Methyl-2-pentanone,0.8755,O=N(=O)C(=C(N)N)N(=O)=O,CS(C)=O,CC(=O)CC(C)C,293.15,1.408334
2,"1,1-Diamino-2,2-dinitroethylene",Dimethyl sulfoxide,4-Methyl-2-pentanone,0.8040,O=N(=O)C(=C(N)N)N(=O)=O,CS(C)=O,CC(=O)CC(C)C,293.15,1.366396
3,"1,1-Diamino-2,2-dinitroethylene",Dimethyl sulfoxide,4-Methyl-2-pentanone,0.7251,O=N(=O)C(=C(N)N)N(=O)=O,CS(C)=O,CC(=O)CC(C)C,293.15,1.305628
4,"1,1-Diamino-2,2-dinitroethylene",Dimethyl sulfoxide,4-Methyl-2-pentanone,0.6375,O=N(=O)C(=C(N)N)N(=O)=O,CS(C)=O,CC(=O)CC(C)C,293.15,1.231354
...,...,...,...,...,...,...,...,...,...
26701,"β-octahydro-1,3,5,7-tetranitro-1,3,5,7-tetrazo...",Dimethyl sulfoxide,Water,0.5000,O=N(=O)N1CN(N(=O)=O)CN(N(=O)=O)CN(N(=O)=O)C1,CS(C)=O,O,368.15,1.463315
26702,"β-octahydro-1,3,5,7-tetranitro-1,3,5,7-tetrazo...",Dimethyl sulfoxide,Water,0.3750,O=N(=O)N1CN(N(=O)=O)CN(N(=O)=O)CN(N(=O)=O)C1,CS(C)=O,O,368.15,1.149765
26703,"β-octahydro-1,3,5,7-tetranitro-1,3,5,7-tetrazo...",Dimethyl sulfoxide,Water,0.2500,O=N(=O)N1CN(N(=O)=O)CN(N(=O)=O)CN(N(=O)=O)C1,CS(C)=O,O,368.15,0.585924
26704,"β-octahydro-1,3,5,7-tetranitro-1,3,5,7-tetrazo...",Dimethyl sulfoxide,Water,0.1250,O=N(=O)N1CN(N(=O)=O)CN(N(=O)=O)CN(N(=O)=O)C1,CS(C)=O,O,368.15,0.049611


In [56]:
final_lab = updated_lab[columns_to_keep]
final_lab

,Drug,Solvent_1,Solvent_2,Solvent_1_mol_fraction,Drug_Canonical_smiles,Solvent_1_Canonical_smiles,Solvent_2_Canonical_smiles,Temperature (K),LogS
0,Celecoxib,Ethanol,Water,0.089083,O=S(=O)(N)C1=CC=C(C=C1)N2N=C(C=C2C=3C=CC(=CC3)...,CCO,O,298.15,-3.061188
1,Celecoxib,Ethanol,Water,0.281185,O=S(=O)(N)C1=CC=C(C=C1)N2N=C(C=C2C=3C=CC(=CC3)...,CCO,O,298.15,-0.351141
2,Celecoxib,Ethanol,Water,0.610093,O=S(=O)(N)C1=CC=C(C=C1)N2N=C(C=C2C=3C=CC(=CC3)...,CCO,O,298.15,0.685046
3,Celecoxib,Ethanol,Water,0.089083,O=S(=O)(N)C1=CC=C(C=C1)N2N=C(C=C2C=3C=CC(=CC3)...,CCO,O,313.15,-2.505811
4,Celecoxib,Ethanol,Water,0.281185,O=S(=O)(N)C1=CC=C(C=C1)N2N=C(C=C2C=3C=CC(=CC3)...,CCO,O,313.15,0.011534
5,Celecoxib,Ethanol,Water,0.610093,O=S(=O)(N)C1=CC=C(C=C1)N2N=C(C=C2C=3C=CC(=CC3)...,CCO,O,313.15,0.933291
6,Acetaminophen,Ethanol,Water,0.089083,O=C(NC1=CC=C(O)C=C1)C,CCO,O,298.15,0.568126
7,Acetaminophen,Ethanol,Water,0.281185,O=C(NC1=CC=C(O)C=C1)C,CCO,O,298.15,1.165377
8,Acetaminophen,Ethanol,Water,0.610093,O=C(NC1=CC=C(O)C=C1)C,CCO,O,298.15,1.280612
9,Acetaminophen,Ethanol,Water,0.089083,O=C(NC1=CC=C(O)C=C1)C,CCO,O,313.15,0.830016


In [ ]:
drugs_list = drugs_df[['Drug', 'Canonical_smiles', 'standardized_SMILES']]
solvents_list = solvents_df[['Solvent', 'Canonical_smiles', 'standardized_SMILES']]
drugs_list = drugs_list.rename(columns={'Canonical_smiles':'smiles_canon', 'standardized_SMILES': 'Standardized_smiles'})
solvents_list = solvents_list.rename(columns={'Canonical_smiles':'smiles_canon','standardized_SMILES': 'Standardized_smiles'})

In [92]:
drugs_list

,Drug,Canonical_smiles,Standardized_smiles
0,Guanidine hydrochloride,Cl.N=C(N)N,Cl.N=C(N)N
1,Glucosamine hydrochloride,Cl.O=CC(N)C(O)C(O)C(O)CO,Cl.NC(C=O)C(O)C(O)C(O)CO
2,2-Amino-6-chloropyrazine,ClC=1N=C(N)C=NC1,Nc1cncc(Cl)n1
3,Thiamine nitrate,O=N(=O)[O-].OCCC=1SC=[N+](C1C)CC2=CN=C(N=C2N)C,Cc1ncc(C[n+]2csc(CCO)c2C)c(N)n1.O=[N+]([O-])[O-]
4,Aripiprazole,O=C1NC2=CC(OCCCCN3CCN(C=4C=CC=C(Cl)C4Cl)CC3)=C...,O=C1CCc2ccc(OCCCCN3CCN(c4cccc(Cl)c4Cl)CC3)cc2N1
...,...,...,...
121,Carbendazim,O=C(OC)NC1=NC=2C=CC=CC2N1,COC(=O)Nc1nc2ccccc2[nH]1
122,Vinpocetine,O=C(OCC)C1=CC2(CC)CCCN3CCC=4C=5C=CC=CC5N1C4C32,CCOC(=O)C1=C[C@]2(CC)CCCN3CCc4c(n1c1ccccc41)[C...
123,3-Indolepropionic acid,O=C(O)CCC1=CNC=2C=CC=CC21,O=C(O)CCc1c[nH]c2ccccc12
124,Hydrocortisone,O=C1C=C2CCC3C4CCC(O)(C(=O)CO)C4(C)CC(O)C3C2(C)CC1,C[C@]12CCC(=O)C=C1CC[C@@H]1[C@@H]2[C@@H](O)C[C...


In [93]:
def add_molecular_descriptor_2_dataframe(df, descriptor, rdkit_fun):
    descriptor_vect = []
    for smiles in df['Canonical_smiles']:
        try:
            descriptor_elem = rdkit_fun(Chem.MolFromSmiles(smiles))
        except:
            descriptor_elem = None
        descriptor_vect.append(descriptor_elem)
    df[descriptor] = descriptor_vect

add_molecular_descriptor_2_dataframe(drugs_list, 'molecular_weight', MolWt)
add_molecular_descriptor_2_dataframe(drugs_list, 'TPSA', CalcTPSA)
add_molecular_descriptor_2_dataframe(solvents_list, 'molecular_weight', MolWt)
add_molecular_descriptor_2_dataframe(solvents_list, 'TPSA', CalcTPSA)
drugs_list

,Drug,Canonical_smiles,Standardized_smiles,molecular_weight,TPSA
0,Guanidine hydrochloride,Cl.N=C(N)N,Cl.N=C(N)N,95.533,75.89
1,Glucosamine hydrochloride,Cl.O=CC(N)C(O)C(O)C(O)CO,Cl.NC(C=O)C(O)C(O)C(O)CO,215.633,124.01
2,2-Amino-6-chloropyrazine,ClC=1N=C(N)C=NC1,Nc1cncc(Cl)n1,129.550,51.80
3,Thiamine nitrate,O=N(=O)[O-].OCCC=1SC=[N+](C1C)CC2=CN=C(N=C2N)C,Cc1ncc(C[n+]2csc(CCO)c2C)c(N)n1.O=[N+]([O-])[O-],327.366,142.11
4,Aripiprazole,O=C1NC2=CC(OCCCCN3CCN(C=4C=CC=C(Cl)C4Cl)CC3)=C...,O=C1CCc2ccc(OCCCCN3CCN(c4cccc(Cl)c4Cl)CC3)cc2N1,448.394,44.81
...,...,...,...,...,...
121,Carbendazim,O=C(OC)NC1=NC=2C=CC=CC2N1,COC(=O)Nc1nc2ccccc2[nH]1,191.190,67.01
122,Vinpocetine,O=C(OCC)C1=CC2(CC)CCCN3CCC=4C=5C=CC=CC5N1C4C32,CCOC(=O)C1=C[C@]2(CC)CCCN3CCc4c(n1c1ccccc41)[C...,350.462,34.47
123,3-Indolepropionic acid,O=C(O)CCC1=CNC=2C=CC=CC21,O=C(O)CCc1c[nH]c2ccccc12,189.214,53.09
124,Hydrocortisone,O=C1C=C2CCC3C4CCC(O)(C(=O)CO)C4(C)CC(O)C3C2(C)CC1,C[C@]12CCC(=O)C=C1CC[C@@H]1[C@@H]2[C@@H](O)C[C...,362.466,94.83


In [ ]:
drugs_list['solute_id'] = [i for i in range(drugs_list.shape[0])]
solvents_list['solvent_id'] = [i for i in range(solvents_list.shape[0])]
drugs_list

,Drug,Canonical_smiles,Standardized_smiles,molecular_weight,TPSA,Drug_id
0,Guanidine hydrochloride,Cl.N=C(N)N,Cl.N=C(N)N,95.533,75.89,0
1,Glucosamine hydrochloride,Cl.O=CC(N)C(O)C(O)C(O)CO,Cl.NC(C=O)C(O)C(O)C(O)CO,215.633,124.01,1
2,2-Amino-6-chloropyrazine,ClC=1N=C(N)C=NC1,Nc1cncc(Cl)n1,129.550,51.80,2
3,Thiamine nitrate,O=N(=O)[O-].OCCC=1SC=[N+](C1C)CC2=CN=C(N=C2N)C,Cc1ncc(C[n+]2csc(CCO)c2C)c(N)n1.O=[N+]([O-])[O-],327.366,142.11,3
4,Aripiprazole,O=C1NC2=CC(OCCCCN3CCN(C=4C=CC=C(Cl)C4Cl)CC3)=C...,O=C1CCc2ccc(OCCCCN3CCN(c4cccc(Cl)c4Cl)CC3)cc2N1,448.394,44.81,4
...,...,...,...,...,...,...
121,Carbendazim,O=C(OC)NC1=NC=2C=CC=CC2N1,COC(=O)Nc1nc2ccccc2[nH]1,191.190,67.01,121
122,Vinpocetine,O=C(OCC)C1=CC2(CC)CCCN3CCC=4C=5C=CC=CC5N1C4C32,CCOC(=O)C1=C[C@]2(CC)CCCN3CCc4c(n1c1ccccc41)[C...,350.462,34.47,122
123,3-Indolepropionic acid,O=C(O)CCC1=CNC=2C=CC=CC21,O=C(O)CCc1c[nH]c2ccccc12,189.214,53.09,123
124,Hydrocortisone,O=C1C=C2CCC3C4CCC(O)(C(=O)CO)C4(C)CC(O)C3C2(C)CC1,C[C@]12CCC(=O)C=C1CC[C@@H]1[C@@H]2[C@@H](O)C[C...,362.466,94.83,124


In [95]:
drugs_list.to_csv('C:/Users/kverg/GDI-NN/data/solubility_cosolvents/drugs_list.csv', index=False)
solvents_list.to_csv('C:/Users/kverg/GDI-NN/data/solubility_cosolvents/sovlents_list.csv', index=False)

In [ ]:
def get_cell_value(colname, row):
    try:
        value = row[colname].values[0]
    except:
        value = None
    return value

solv1_id = []
solv2_id = []
solute_id = []
smiles_canon_solv1 = []
smiles_canon_solv2 = []
smiles_canon_solute = []
polarity_solv1 = []
polarity_solv2 = []
polarity_solute = []
for i in range(systems_set.shape[0]):
    row = systems_set.iloc[i]
    solute_name = row['solute']
    solvent1_name = row['solvent_1']
    solvent2_name = row['solvent_2']
    
    solute_row = solute_set[solute_set['compoundName']==solute_name]
    solvent1_row = solvent_set[solvent_set['compoundName']==solvent1_name]
    solvent2_row = solvent_set[solvent_set['compoundName']==solvent2_name]
    
    solv1_id.append(get_cell_value('compoundID', solvent1_row))
    solv2_id.append(get_cell_value('compoundID', solvent2_row))
    solute_id.append(get_cell_value('compoundID', solute_row))
    smiles_canon_solv1.append(get_cell_value('smiles_canon', solvent1_row))
    smiles_canon_solv2.append(get_cell_value('smiles_canon', solvent2_row))
    smiles_canon_solute.append(get_cell_value('smiles_canon', solute_row))
    polarity_solv1.append(get_cell_value('TPSA', solvent1_row))
    polarity_solv2.append(get_cell_value('TPSA', solvent2_row))
    polarity_solute.append(get_cell_value('TPSA', solute_row))

    if i % 10000 ==0:
        print(i)

In [ ]:
train_test_df = pd.concat([final_literature, final_lab], axis=0)
train_test_df_filtered_Temp = train_test_df[train_test_df['Temperature (K)'] == 298.15]
train_test_df_filtered_Temp_and_molfrac = train_test_df_filtered_Temp[train_test_df_filtered_Temp['Solvent_1_mol_fraction'] >= 0.1]
train_test_df_filtered_Temp_and_molfrac = train_test_df_filtered_Temp_and_molfrac[train_test_df_filtered_Temp_and_molfrac['Solvent_1_mol_fraction'] <= 0.9]
train_test_df_filtered_Temp_and_molfrac

In [ ]:
train_test_df.to_csv('C:/Users/kverg/GDI-NN/data/solubility_cosolvents/systems_train_temp_not_filtered.csv', index=False)
train_test_df_filtered_Temp.to_csv('C:/Users/kverg/GDI-NN/data/solubility_cosolvents/systems_train_298K.csv', index=False)
train_test_df_filtered_Temp_and_molfrac.to_csv('C:/Users/kverg/GDI-NN/data/solubility_cosolvents/systems_train_298K_0109molfrac.csv', index=False)

In [ ]:
train_test_df_filtered_Temp_and_molfrac = pd.read_csv('C:/Users/kverg/GDI-NN/data/solubility_cosolvents/systems_train_298K_0109molfrac.csv')
train_test_df_filtered_Temp_and_molfrac['Water'] = ['Water']*train_test_df_filtered_Temp_and_molfrac.shape[0]
train_test_df_filtered_Temp_and_molfrac['Water_smiles'] = ['O']*train_test_df_filtered_Temp_and_molfrac.shape[0]
#drug_water_system_actcoef = pd.read_csv('C:/Users/kverg/GDI-NN/data/solubility_cosolvents/benchmark_actcoef_drugs_water_systems.csv')

train_test_df_filtered_Temp_and_molfrac.to_csv('C:/Users/kverg/GDI-NN/data/solubility_cosolvents/systems_train_298K_0109molfrac.csv')